In [ ]:
import pydicom
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

IMAGES_DIR = Path("../data/raw/tcia/qin_prostate_repeatability/images")

SEG_SOP = "1.2.840.10008.5.1.4.1.1.66.4"
MR_SOP  = "1.2.840.10008.5.1.4.1.1.4"

SEGMENT_COLORS = {
    "WholeGland":    "#4C72B0",
    "PeripheralZone": "#DD8452",
    "NormalROI_PZ_1": "#2CA02C",
    "TumorROI_PZ_1":  "#D62728",
}

## Series inventory
Read one DICOM header per series directory to build a summary table.

In [ ]:
rows = []
for series_dir in tqdm(sorted(IMAGES_DIR.iterdir())):
    dcm_files = sorted(series_dir.glob("*.dcm"))
    if not dcm_files:
        continue
    ds = pydicom.dcmread(dcm_files[0], stop_before_pixels=True)
    rows.append({
        "series_dir":        series_dir.name,
        "SeriesInstanceUID": str(ds.get("SeriesInstanceUID", "")),
        "PatientID":         str(ds.get("PatientID", "")),
        "StudyInstanceUID":  str(ds.get("StudyInstanceUID", "")),
        "StudyUID_short":    str(ds.get("StudyInstanceUID", ""))[:24],
        "Modality":          str(ds.get("Modality", "")),
        "SeriesDescription": str(ds.get("SeriesDescription", "")),
        "TimePoint":         str(ds.get("ClinicalTrialTimePointID", "")),
        "StudyDate":         str(ds.get("StudyDate", "")),
        "Rows":              ds.get("Rows", None),
        "Columns":           ds.get("Columns", None),
        "n_files":           len(dcm_files),
    })

df = pd.DataFrame(rows)
print(f"{len(df)} series found")
df.head()

In [ ]:
# Summary: series count per patient × modality × description
df.groupby(["PatientID", "Modality", "SeriesDescription"]).size().rename("count").reset_index()

## Visit structure and SEG→MR linkage
Each patient has 2 visits distinguished by `ClinicalTrialTimePointID` (1 or 2) and a different `StudyInstanceUID`. SEGs are linked to their specific MR via `ReferencedSeriesSequence[0].SeriesInstanceUID`.

In [ ]:
# ClinicalTrialTimePointID is set on SEG/SR but not MR series.
# Build a full StudyInstanceUID -> TimePoint mapping from series that carry it,
# then propagate to all series sharing the same study UID.
study_to_tp = (
    df[df["TimePoint"] != ""]
    .drop_duplicates("StudyInstanceUID")
    .set_index("StudyInstanceUID")["TimePoint"]
    .to_dict()
)
df["TimePoint"] = df.apply(
    lambda r: r["TimePoint"] if r["TimePoint"] != "" else study_to_tp.get(r["StudyInstanceUID"], ""),
    axis=1,
)

# Part 1: visit structure — pivot PatientID × TimePoint → StudyDate
visit_df = (
    df[df["Modality"] == "MR"]
    .drop_duplicates(subset=["PatientID", "TimePoint"])
    [["PatientID", "TimePoint", "StudyDate"]]
    .sort_values(["PatientID", "TimePoint"])
)
pivot = visit_df.pivot(index="PatientID", columns="TimePoint", values="StudyDate")
pivot.columns = [f"Visit {c} date" for c in pivot.columns]
display(pivot)

# Part 2: SEG→MR linkage for one subject
subj = "PCAMPMRI-00001"
link_rows = []
for _, seg_row in df[(df["PatientID"] == subj) & (df["Modality"] == "SEG")].iterrows():
    dcm = pydicom.dcmread(
        next((IMAGES_DIR / seg_row["series_dir"]).glob("*.dcm")),
        stop_before_pixels=True,
    )
    ref_uid = str(dcm.ReferencedSeriesSequence[0].SeriesInstanceUID)
    mr = df[df["SeriesInstanceUID"] == ref_uid]
    link_rows.append({
        "SEG description": seg_row["SeriesDescription"],
        "SEG TimePoint":   seg_row["TimePoint"],
        "→ MR description": mr.iloc[0]["SeriesDescription"] if not mr.empty else "NOT FOUND",
        "MR TimePoint":    mr.iloc[0]["TimePoint"]          if not mr.empty else "?",
        "link tag":        "ReferencedSeriesSequence[0].SeriesInstanceUID",
    })

print(f"\nSEG→MR linkage for {subj}:")
display(pd.DataFrame(link_rows).sort_values(["SEG TimePoint", "SEG description"]))

## SEG series present per patient

In [ ]:
seg_df = df[df["Modality"] == "SEG"].copy().reset_index(drop=True)
print(f"{len(seg_df)} SEG series\n")
print(seg_df["SeriesDescription"].value_counts().to_string())
display(seg_df[["PatientID", "SeriesDescription", "Rows", "Columns", "n_files"]])

## Segment descriptions inside each SEG file

In [ ]:
seg_info = []
for _, row in seg_df.iterrows():
    dcm_path = next((IMAGES_DIR / row["series_dir"]).glob("*.dcm"))
    ds = pydicom.dcmread(dcm_path, stop_before_pixels=True)
    for seg in ds.SegmentSequence:
        category = ""
        if hasattr(seg, "SegmentedPropertyCategoryCodeSequence"):
            category = seg.SegmentedPropertyCategoryCodeSequence[0].CodeMeaning
        seg_info.append({
            "PatientID":          row["PatientID"],
            "SeriesDescription":  row["SeriesDescription"],
            "SegmentNumber":      int(seg.SegmentNumber),
            "SegmentDescription": str(seg.SegmentDescription),
            "Category":           category,
        })

seg_info_df = pd.DataFrame(seg_info)
display(seg_info_df)
print("\nUnique segment descriptions:", seg_info_df["SegmentDescription"].unique())

## Dimension check: SEG vs referenced MR series

In [ ]:
# Build lookup: SeriesInstanceUID -> (Rows, Columns)
uid_to_dims = {
    row["SeriesInstanceUID"]: (row["Rows"], row["Columns"])
    for _, row in df.iterrows()
}

dim_check = []
for _, row in seg_df.iterrows():
    dcm_path = next((IMAGES_DIR / row["series_dir"]).glob("*.dcm"))
    ds = pydicom.dcmread(dcm_path, stop_before_pixels=True)
    ref_uid = ""
    if hasattr(ds, "ReferencedSeriesSequence"):
        ref_uid = str(ds.ReferencedSeriesSequence[0].SeriesInstanceUID)
    mr_dims = uid_to_dims.get(ref_uid, (None, None))
    seg_dims = (row["Rows"], row["Columns"])
    dim_check.append({
        "PatientID":         row["PatientID"],
        "SeriesDescription": row["SeriesDescription"],
        "SEG_Rows":          seg_dims[0],
        "SEG_Cols":          seg_dims[1],
        "MR_Rows":           mr_dims[0],
        "MR_Cols":           mr_dims[1],
        "ref_found":         ref_uid in uid_to_dims,
        "dims_match":        seg_dims == mr_dims,
    })

dim_df = pd.DataFrame(dim_check)
display(dim_df)
n_match = dim_df["dims_match"].sum()
n_ref_missing = (~dim_df["ref_found"]).sum()
print(f"\nDimensions match: {n_match}/{len(dim_df)}")
if n_ref_missing:
    print(f"Referenced MR series not found in inventory: {n_ref_missing} SEG(s)")

## Visualize: T2 MR slices with SEG overlays
Pick the first patient's T2 axial series and its corresponding T2 segmentation.

In [ ]:
def load_mr_volume(series_dir: Path):
    """Load a single-series MR stack sorted by InstanceNumber. Returns (3D array H×W×Z, spacing)."""
    slices = []
    for f in sorted(series_dir.glob("*.dcm")):
        ds = pydicom.dcmread(f)
        slices.append((int(ds.get("InstanceNumber", 0)), ds.pixel_array, ds))
    slices.sort(key=lambda x: x[0])
    vol = np.stack([s[1] for s in slices], axis=-1)  # H × W × Z
    ds0 = slices[0][2]
    px_sp = [float(v) for v in ds0.PixelSpacing]
    sl_th = float(ds0.get("SliceThickness", 1.0))
    return vol, (px_sp[0], px_sp[1], sl_th)


def load_seg_volume(seg_dcm_path: Path):
    """Load a DICOM SEG file. Returns dict {SegmentDescription: 3D bool array H×W×Z}."""
    ds = pydicom.dcmread(seg_dcm_path)
    # pixel_array shape: (n_frames, H, W)
    px = ds.pixel_array  # 0/1 per frame
    n_frames = px.shape[0]

    # Map each frame index to its segment number
    frame_to_seg = {}
    per_frame = ds.PerFrameFunctionalGroupsSequence
    for i, fg in enumerate(per_frame):
        seg_num = int(fg.SegmentIdentificationSequence[0].ReferencedSegmentNumber)
        frame_to_seg[i] = seg_num

    seg_num_to_desc = {int(s.SegmentNumber): str(s.SegmentDescription) for s in ds.SegmentSequence}

    # Group frames by segment
    seg_frames = {}
    for frame_idx, seg_num in frame_to_seg.items():
        seg_frames.setdefault(seg_num, []).append((frame_idx, frame_idx))  # (frame_idx, slice_pos)

    result = {}
    for seg_num, frames in seg_frames.items():
        desc = seg_num_to_desc.get(seg_num, str(seg_num))
        arr = np.zeros((px.shape[1], px.shape[2], len(frames)), dtype=bool)
        for z, (fi, _) in enumerate(sorted(frames)):
            arr[:, :, z] = px[fi].astype(bool)
        result[desc] = arr
    return result

In [ ]:
# Pick first patient with a T2 SEG
t2_seg_rows = seg_df[seg_df["SeriesDescription"].str.contains("T2", case=False)]
example_seg_row = t2_seg_rows.iloc[0]
patient_id = example_seg_row["PatientID"]
print(f"Visualising: {patient_id} — {example_seg_row['SeriesDescription']}")

# Find the referenced MR series
seg_dcm_path = next((IMAGES_DIR / example_seg_row["series_dir"]).glob("*.dcm"))
seg_ds_hdr = pydicom.dcmread(seg_dcm_path, stop_before_pixels=True)
ref_series_uid = str(seg_ds_hdr.ReferencedSeriesSequence[0].SeriesInstanceUID)

mr_row = df[df["SeriesInstanceUID"] == ref_series_uid].iloc[0]
print(f"Referenced MR: {mr_row['SeriesDescription']} ({mr_row['n_files']} slices)")

In [ ]:
mr_vol, mr_spacing = load_mr_volume(IMAGES_DIR / mr_row["series_dir"])
seg_masks = load_seg_volume(seg_dcm_path)
print(f"MR volume shape: {mr_vol.shape}, spacing: {mr_spacing}")
for desc, arr in seg_masks.items():
    print(f"  SEG '{desc}': {arr.shape}, non-zero slices: {arr.any(axis=(0,1)).sum()}")

In [ ]:
def show_mr_with_seg(mr_vol, seg_masks, n_slices=8, title=""):
    nz = mr_vol.shape[2]
    # Focus on slices that have at least one non-empty SEG mask
    any_mask = np.zeros(nz, dtype=bool)
    for arr in seg_masks.values():
        n = min(arr.shape[2], nz)
        any_mask[:n] |= arr[:, :, :n].any(axis=(0, 1))
    active = np.where(any_mask)[0]
    if len(active) == 0:
        active = np.arange(nz)
    indices = active[np.linspace(0, len(active) - 1, min(n_slices, len(active)), dtype=int)]

    vmin, vmax = np.percentile(mr_vol, [1, 99])
    fig, axes = plt.subplots(1, len(indices), figsize=(2.5 * len(indices), 3.5))
    if len(indices) == 1:
        axes = [axes]
    fig.suptitle(title, y=1.01)

    for ax, sl in zip(axes, indices):
        ax.imshow(mr_vol[:, :, sl].T, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
        for desc, arr in seg_masks.items():
            if sl >= arr.shape[2]:
                continue
            mask = arr[:, :, sl]
            if not mask.any():
                continue
            color = SEGMENT_COLORS.get(desc, "#888888")
            overlay = np.zeros((*mr_vol[:, :, sl].shape, 4))
            overlay[..., :3] = plt.matplotlib.colors.to_rgb(color)
            overlay[..., 3] = mask.astype(float) * 0.45
            ax.imshow(overlay.transpose(1, 0, 2), origin="lower")
        ax.set_title(f"z={sl}", fontsize=8)
        ax.axis("off")

    handles = [
        plt.matplotlib.patches.Patch(color=SEGMENT_COLORS.get(d, "#888"), label=d)
        for d in seg_masks
    ]
    fig.legend(handles=handles, loc="lower center", ncol=len(handles), bbox_to_anchor=(0.5, -0.08), fontsize=8)
    plt.tight_layout()
    plt.show()


show_mr_with_seg(mr_vol, seg_masks, title=f"{patient_id} — {example_seg_row['SeriesDescription']}")

## Batch: one row per patient, T2 middle slice with all segment overlays

In [ ]:
t2_seg_rows = seg_df[seg_df["SeriesDescription"].str.contains("T2", case=False)].copy()
n = len(t2_seg_rows)
fig, axes = plt.subplots(n, 1, figsize=(12, 3 * n), squeeze=False)

for row_idx, (_, seg_row) in enumerate(t2_seg_rows.iterrows()):
    ax = axes[row_idx, 0]
    try:
        s_dcm = next((IMAGES_DIR / seg_row["series_dir"]).glob("*.dcm"))
        s_hdr = pydicom.dcmread(s_dcm, stop_before_pixels=True)
        ref_uid = str(s_hdr.ReferencedSeriesSequence[0].SeriesInstanceUID)
        mr_rows = df[df["SeriesInstanceUID"] == ref_uid]
        if mr_rows.empty:
            ax.text(0.5, 0.5, "MR not found", ha="center", va="center", transform=ax.transAxes)
            ax.axis("off")
            continue
        mr_vol, _ = load_mr_volume(IMAGES_DIR / mr_rows.iloc[0]["series_dir"])
        masks = load_seg_volume(s_dcm)
        sl = mr_vol.shape[2] // 2
        vmin, vmax = np.percentile(mr_vol, [1, 99])
        ax.imshow(mr_vol[:, :, sl].T, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
        for desc, arr in masks.items():
            if sl >= arr.shape[2]:
                continue
            mask = arr[:, :, sl]
            if not mask.any():
                continue
            color = SEGMENT_COLORS.get(desc, "#888")
            overlay = np.zeros((*mr_vol[:, :, sl].shape, 4))
            overlay[..., :3] = plt.matplotlib.colors.to_rgb(color)
            overlay[..., 3] = mask.astype(float) * 0.45
            ax.imshow(overlay.transpose(1, 0, 2), origin="lower")
        ax.set_title(f"{seg_row['PatientID']} — {seg_row['SeriesDescription']} z={sl}", fontsize=9)
    except Exception as e:
        ax.text(0.5, 0.5, str(e), ha="center", va="center", transform=ax.transAxes, fontsize=7)
    ax.axis("off")

handles = [plt.matplotlib.patches.Patch(color=c, label=d) for d, c in SEGMENT_COLORS.items()]
fig.legend(handles=handles, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.01), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
df.columns

In [ ]:
with pd.option_context('display.max_rows', None):
    display(df.sort_values(["PatientID", "SeriesDescription", "Modality"])[['PatientID', 'Modality', 'SeriesDescription', 'Rows', 'Columns', 'n_files']])

## SR measurements: volume per segment per patient

In [ ]:
def parse_sr_measurements(ds):
    """Extract (tracking_identifier, volume_mm3) pairs from an Enhanced SR dataset."""
    results = []
    if not hasattr(ds, "ContentSequence"):
        return results
    for item in ds.ContentSequence:
        if str(item.get("ValueType", "")) != "CONTAINER":
            continue
        name = ""
        if hasattr(item, "ConceptNameCodeSequence"):
            name = item.ConceptNameCodeSequence[0].CodeMeaning
        if name != "Imaging Measurements":
            continue
        for group in item.ContentSequence:
            tracking_id = ""
            volume = None
            for child in group.ContentSequence:
                cname = ""
                if hasattr(child, "ConceptNameCodeSequence"):
                    cname = child.ConceptNameCodeSequence[0].CodeMeaning
                if cname == "Tracking Identifier":
                    tracking_id = str(child.get("TextValue", ""))
                if cname == "Volume" and str(child.get("ValueType", "")) == "NUM":
                    mv = child.MeasuredValueSequence[0]
                    volume = float(mv.NumericValue)
            if tracking_id and volume is not None:
                results.append((tracking_id, volume))
    return results


sr_df_rows = []
sr_series = df[df["Modality"] == "SR"]
for _, row in sr_series.iterrows():
    dcm_path = next((IMAGES_DIR / row["series_dir"]).glob("*.dcm"))
    ds = pydicom.dcmread(dcm_path)
    for tracking_id, volume_mm3 in parse_sr_measurements(ds):
        sr_df_rows.append({
            "PatientID":         row["PatientID"],
            "SeriesDescription": row["SeriesDescription"],
            "Segment":           tracking_id.replace(" measurements", ""),
            "Volume_mm3":        volume_mm3,
            "Volume_mL":         volume_mm3 / 1000,
        })

sr_meas = pd.DataFrame(sr_df_rows)
print(f"{len(sr_series)} SR series, {len(sr_meas)} measurements")
display(sr_meas.head(12))

In [ ]:
t2_meas = sr_meas[sr_meas["SeriesDescription"].str.contains("T2", case=False)].copy()
segments = t2_meas["Segment"].unique()
patients = sorted(t2_meas["PatientID"].unique())

fig, axes = plt.subplots(1, len(segments), figsize=(4 * len(segments), 5), sharey=False)
if len(segments) == 1:
    axes = [axes]

for ax, seg in zip(axes, segments):
    subset = t2_meas[t2_meas["Segment"] == seg].sort_values("PatientID")
    color = SEGMENT_COLORS.get(seg, "#888888")
    ax.bar(range(len(subset)), subset["Volume_mL"], color=color, alpha=0.8)
    ax.set_xticks(range(len(subset)))
    ax.set_xticklabels(subset["PatientID"].str.replace("PCAMPMRI-", ""), rotation=90, fontsize=7)
    ax.set_title(seg, fontsize=9)
    ax.set_ylabel("Volume (mL)")
    ax.set_xlabel("Patient")

fig.suptitle("T2 SR Measurements: volume per segment", y=1.02)
plt.tight_layout()
plt.show()

## All SEG types for one subject on the corresponding MR

In [ ]:
SUBJECT = "PCAMPMRI-00001"
N_SLICES = 6

subject_segs = df[(df["PatientID"] == SUBJECT) & (df["Modality"] == "SEG")].copy()

# One row per unique SEG type (first visit if duplicates)
seg_types = subject_segs.drop_duplicates(subset="SeriesDescription", keep="first")

n_rows = len(seg_types)
fig, axes = plt.subplots(n_rows, N_SLICES, figsize=(2.5 * N_SLICES, 3.2 * n_rows))
if n_rows == 1:
    axes = axes[np.newaxis, :]

for row_idx, (_, seg_row) in enumerate(seg_types.iterrows()):
    seg_dcm = next((IMAGES_DIR / seg_row["series_dir"]).glob("*.dcm"))

    # Find referenced MR
    seg_hdr = pydicom.dcmread(seg_dcm, stop_before_pixels=True)
    ref_uid = str(seg_hdr.ReferencedSeriesSequence[0].SeriesInstanceUID)
    mr_match = df[df["SeriesInstanceUID"] == ref_uid]
    if mr_match.empty:
        for ax in axes[row_idx]:
            ax.text(0.5, 0.5, "MR not found", ha="center", va="center", transform=ax.transAxes)
            ax.axis("off")
        continue

    mr_vol, _ = load_mr_volume(IMAGES_DIR / mr_match.iloc[0]["series_dir"])
    seg_masks = load_seg_volume(seg_dcm)

    # Slices with any mask content
    nz = mr_vol.shape[2]
    seg_z = min(min(a.shape[2] for a in seg_masks.values()), nz)
    active = np.where(
        np.stack([a[:, :, :seg_z].any(axis=(0, 1)) for a in seg_masks.values()]).any(axis=0)
    )[0]
    if len(active) == 0:
        active = np.arange(nz)
    indices = active[np.linspace(0, len(active) - 1, N_SLICES, dtype=int)]

    vmin, vmax = np.percentile(mr_vol, [1, 99])
    mr_desc = mr_match.iloc[0]["SeriesDescription"]

    for col_idx, sl in enumerate(indices):
        ax = axes[row_idx, col_idx]
        ax.imshow(mr_vol[:, :, sl].T, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
        for desc, arr in seg_masks.items():
            if sl >= arr.shape[2]:
                continue
            mask = arr[:, :, sl]
            if not mask.any():
                continue
            color = SEGMENT_COLORS.get(desc, "#888888")
            overlay = np.zeros((*mr_vol[:, :, sl].shape, 4))
            overlay[..., :3] = plt.matplotlib.colors.to_rgb(color)
            overlay[..., 3] = mask.astype(float) * 0.45
            ax.imshow(overlay.transpose(1, 0, 2), origin="lower")
        ax.set_title(f"z={sl}", fontsize=7)
        ax.axis("off")
        if col_idx == 0:
            ax.set_ylabel(f"{seg_row['SeriesDescription']}\n(MR: {mr_desc})", fontsize=7)

# Shared legend
handles = [plt.matplotlib.patches.Patch(color=c, label=d) for d, c in SEGMENT_COLORS.items()]
fig.legend(handles=handles, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.02), fontsize=8)
fig.suptitle(f"{SUBJECT} — all SEG types on corresponding MR", y=1.01, fontsize=10)
plt.tight_layout()
plt.show()